In [1]:
import pandas as pd
from tqdm.notebook import tqdm
from pprint import pprint
import tokenizer
import importlib
importlib.reload(tokenizer)
import json
import torch
import torch.nn as nn

In [2]:
df_train = pd.read_parquet('data/train_wiki_cleaned_cutted.pq')
df_eval = pd.read_parquet('data/eval_wiki_cleaned_cutted.pq')
df_test = pd.read_parquet('data/test_wiki_cleaned_cutted.pq')

In [3]:
with open("artifacts/character_tokenizer.json", "r", encoding="utf-8") as f:
    dict_json = json.load(f)

In [4]:
tok = tokenizer.Tokenizer.from_dict(dict_json)

In [5]:
class WikipediaDataset(torch.utils.data.Dataset):
    def __init__(self, texts: list, tokenizer: tokenizer.Tokenizer, max_len: int):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, i):
        tokens = self.tokenizer.encode(self.texts[i])
        tokens = torch.tensor(tokens, dtype=torch.long)
        tokens = tokens[:self.max_len]

        if(len(tokens) < self.max_len):
            n_pad_fill = self.max_len - len(tokens)
            pad_fill = torch.tensor([tok.pad_token_id]*n_pad_fill, dtype=torch.long)
            tokens = torch.concat([tokens, pad_fill])

        input_tokens = tokens[:-1]
        target_tokens = tokens[1:]

        return input_tokens, target_tokens


In [6]:
dataset_train = WikipediaDataset(
    df_train.text_cut.values.tolist(),
    tok,
    max_len=200
)

dataset_eval = WikipediaDataset(
    df_eval.text_cut.values.tolist(),
    tok,
    max_len=200
)

In [7]:
config_train = {
    'batch_size': 64,
    'eval_batch_size': 64,
    'accum_steps': 1,
    'eval_acum_steps': 1,
    'epochs': 100,
    'eval_step': 1000,
    'earlying_stop_criteria': 250,
    'log_steps': 250,
    'embed_size': 256,
    'hidden_size': 256*2,
    'num_layers': 4,
    'learning_rate': 1e-4,
    'name_exp': 'train_v1',
    'desc': 'Tamanho ok',
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

step_info = {'accum_steps': 0, 'eval_accum_steps': 0, 'global_step': 0, 'loss_sum': 0, 'eval_loss_sum': 0, 'best_eval_loss': torch.inf}

In [8]:
import os
os.makedirs(f'models/{config_train['name_exp']}', exist_ok=True)

In [9]:
with open(f"models/{config_train['name_exp']}/config.json", "w", encoding="utf-8") as f:
    json.dump(config_train, f, ensure_ascii=False, indent=2)

In [10]:
dataloader_train = torch.utils.data.DataLoader(
        dataset=dataset_train, 
        batch_size=config_train['batch_size'],
        shuffle=True,
)

dataloader_eval = torch.utils.data.DataLoader(
        dataset=dataset_eval, 
        batch_size=config_train['batch_size'],
        shuffle=True
)

In [11]:
def train_step(config, step_info, model, optimizer, batch, vocab_size):
    model.train()
    input_tokens, output_tokens = batch
    input_tokens = input_tokens.to(config['device'])
    output_tokens = output_tokens.to(config['device'])
    
    states = None
    predict_tokens, states = model(input_tokens, states)

    predict_tokens = predict_tokens.view(-1, vocab_size)
    output_tokens = output_tokens.view(-1)

    loss = nn.functional.cross_entropy(predict_tokens, output_tokens, ignore_index=0)

    (loss / config['accum_steps']).backward()

    step_info['accum_steps'] += 1
    step_info['loss_sum'] += loss.item()

    if(states):
        states = (states[0].detach(), states[1].detach())

    if(step_info['accum_steps'] % config_train['accum_steps'] == 0):
        optimizer.step()
        step_info['global_step'] += 1

        optimizer.zero_grad()

def val_step(config, step_info, model, dataloader, vocab_size):
    model.eval()
    for batch in tqdm(dataloader, total=len(dataloader), desc='Evaluating'):
        input_tokens, output_tokens = batch
        input_tokens = input_tokens.to(config['device'])
        output_tokens = output_tokens.to(config['device'])

        states = None
        predict_tokens, states = model(input_tokens, states)

        predict_tokens = predict_tokens.view(-1, vocab_size)
        output_tokens = output_tokens.view(-1)

        loss = nn.functional.cross_entropy(predict_tokens, output_tokens, ignore_index=0)

        step_info['eval_accum_steps'] += 1
        step_info['eval_loss_sum'] += loss.item()

        if(states):
            states = (states[0].detach(), states[1].detach())

In [12]:
def train(config, step_info, model, optimizer, writer, dataloader_train, dataloader_eval, vocab_size):
    for epoch in range(config['epochs']):
        for batch in tqdm(dataloader_train, total=len(dataloader_train), desc='Training'):
            train_step(config, step_info, model, optimizer, batch, vocab_size)
            if(step_info['global_step'] % config['log_steps'] == 0):
                avg_loss = step_info['loss_sum'] / step_info['accum_steps']

                writer.add_scalar('loss/train', avg_loss, step_info['global_step'])
                step_info['loss_sum'] = 0
                step_info['accum_steps'] = 0

            if(step_info['global_step'] % config['eval_step'] == 0):
                val_step(config, step_info, model, dataloader_eval, vocab_size)

                avg_loss_val = step_info['eval_loss_sum'] / step_info['eval_accum_steps']
                if(avg_loss_val < step_info['best_eval_loss']):
                    step_info['best_eval_loss'] = avg_loss_val
                    torch.save(model.state_dict(), f"models/{config_train['name_exp']}/model.pt")
                
                writer.add_scalar('loss/eval', avg_loss_val, step_info['global_step'])
                step_info['eval_loss_sum'] = 0
                step_info['eval_accum_steps'] = 0

In [13]:
import rnn
from torch.utils.tensorboard import SummaryWriter

In [14]:
model = rnn.MyModel(
    tok.vocab_size, 
    config_train['embed_size'], 
    config_train['hidden_size'], 
    config_train['num_layers']
    ).to(config_train['device'])
optimizer = torch.optim.Adam(model.parameters(), lr=config_train['learning_rate'])
writer = SummaryWriter(log_dir=f'runs/{config_train['name_exp']}')

In [15]:
train(config_train, step_info, model, optimizer, writer, dataloader_train, dataloader_eval, tok.vocab_size)

Training:   0%|          | 0/10303 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/194 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/194 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/194 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/194 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/194 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/194 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/194 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/194 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), f"models/{config_train['name_exp']}.pt")

In [ ]:
def predicting(initial_text: str, model, max_len, tokenizer: tokenizer.Tokenizer, eos_penalty: float, temperature: float):
    model.eval()
    enc_text_atual = tok.encode(initial_text)[:-1]
    enc_text_atual = torch.tensor(enc_text_atual, dtype=torch.long).to('cuda')
    states = None
    for _ in range(max_len):
        logits, states = model(enc_text_atual, states)
        logits = logits[-1, :]

        logits[tok.eos_token_id] *= eos_penalty

        logits = logits / temperature

        next_token = torch.argmax(logits, dim=-1).unsqueeze(dim=0)

        if(int(next_token.item()) == tok.eos_token_id):
            break

        enc_text_atual = torch.concat([enc_text_atual, next_token])

    return tok.decode(enc_text_atual.tolist())


In [ ]:
initial_text = 'São Paulo é'
predicting(initial_text, model, 256, tokenizer, 0.4, 0.2)

'<bos>São Paulo é um dos principais expoentes do Brasil e na Região Metropolitana de São Paulo USP e no município de São Paulo Alegrete em São Paulo USP e tem capacidade para 11 000 pessoas em 2016 na cidade de São Paulo 600 espectadores 200 voltas em 2019 a 2015 e 2017-20'